In [ ]:
!pip install torch==2.6.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124
!pip install -r requirements.txt

In [32]:
import pandas as pd
import sys
import os
sys.modules.pop("data_preparation", None)
from data_preparation import FingerprintDataset
from torch.utils.data import DataLoader
from lightgbm import LGBMClassifier
import os
from tqdm import tqdm
import pyarrow as pa
import pyarrow.parquet as pq

In [25]:
def train_model_for_protein(protein_data, protein_name, batch_size=1000, lgb_params=None):
    """
    Train a LightGBM model for a protein using PyTorch DataLoader batching

    Args:
        protein_data (pd.DataFrame): Data containing SMILES strings and labels
        protein_name (str): Name of the protein
        batch_size (int): Number of samples per batch
        lgb_params (dict): Parameters for the LightGBM classifier

    Returns:
        LGBMClassifier: Trained LightGBM model with the best iteration
    """
    # Initialize Dataset & DataLoader
    dataset = FingerprintDataset(protein_data['molecule_smiles'].tolist(), protein_data['binds'].tolist())
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=20)

    # Initialize LightGBM Classifier
    lgb_cls = LGBMClassifier(**lgb_params)

    # Train model in batches
    for X_batch, y_batch in tqdm(dataloader, desc=f"Training {protein_name}"):
        X_batch = X_batch.numpy()
        y_batch = y_batch.numpy()

        # Convert to Pandas DataFrame
        num_features = X_batch.shape[1]
        df_X_batch = pd.DataFrame(X_batch, columns=[f"feature_{j}" for j in range(num_features)])
        df_y_batch = pd.DataFrame(y_batch, columns=["label"])

        # Train LightGBM
        lgb_cls.fit(
            df_X_batch, 
            df_y_batch.values.ravel(), 
            eval_metric='auc', 
            init_model=lgb_cls.booster_ if hasattr(lgb_cls, "booster_") else None
        )

    # Get the best iteration for early stopping
    best_iteration = lgb_cls.best_iteration_
    print(f"Best iteration for {protein_name}: {best_iteration}")

    # Set the final model to the best iteration
    lgb_cls.set_params(n_estimators=best_iteration)

    return lgb_cls


def save_models(model, protein, save_dir='/models'):
    """
    Save trained LightGBM model to disk

    Args:
        model (LGBMClassifier): Trained LightGBM model
        protein (str): Name of the protein
        save_dir (str): Directory to save the model
    """
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f"{protein}_lightgbm.txt")

    if hasattr(model, "booster_"):
        model.booster_.save_model(model_path)
        print(f"LightGBM model for {protein} saved at {model_path}")
    else:
        print(f"Error: No booster found for {protein} model.")


def train_models_by_protein(batch_size=100000, save_dir='/models', lgb_params=None):
    """
    Train models for multiple proteins and save them

    Args:
        batch_size (int): Number of samples per batch
        save_dir (str): Directory to save models
        lgb_params (dict): LightGBM parameters

    Returns:
        str: Confirmation message
    """
    protein_names = ['sEH', 'BRD4', 'HSA']

    for protein in tqdm(protein_names, desc="Training models"):
        
        train_data_path = f'./train_data/{protein}/{protein}_train.parquet'
      
        train_data = pd.read_parquet(train_data_path)

        model = train_model_for_protein(train_data, protein, batch_size, lgb_params)
        save_models(model, protein, save_dir)

    return "Training complete"

In [ ]:
lgb_params = {
        'max_depth': 11,
        'bagging_fraction': 0.9,
        'learning_rate': 0.05,
        'colsample_bytree': 1,
        'colsample_bynode': 0.5,
        'lambda_l1': 1,
        'objective': 'binary',
        'lambda_l2': 1.5,
        'num_leaves': 490,
        'min_data_in_leaf': 50,
        'verbose': -1,
        'metric': 'average_precision',
        'device': 'gpu'
    }

train_models_by_protein(lgb_params=lgb_params)